# Apache Beam Lab 4: I/O Connectors - Solution

## Overview
This notebook contains the solution for Lab 4: I/O Connectors.

## Solution Implementation

In [ ]:
import apache_beam as beam
import os

def text_io_example():
    sample_file = '/tmp/sample_text.txt'
    with open(sample_file, 'w') as f:
        f.write('hello world\n')
        f.write('apache beam\n')
        f.write('data processing\n')
    
    with beam.Pipeline() as pipeline:
        (
            pipeline
            | 'Read text' >> beam.io.ReadFromText(sample_file)
            | 'Process' >> beam.Map(str.upper)
            | 'Write output' >> beam.io.WriteToText('/tmp/output_text')
        )
    
    print("Text I/O example completed")

print("File-based I/O: Read from and write to various file formats")

## Exercise 1: CSV File Processing

In [ ]:
def csv_processing_pipeline():
    sales_file = '/home/ramdov/projects/beam-code-practice/data/sample_sales.csv'
    output_file = '/tmp/sales_summary.csv'
    
    if not os.path.exists(sales_file):
        print("Sales data not found. Run Lab 0 first.")
        return
    
    import pandas as pd
    df = pd.read_csv(sales_file)
    data = df.to_dict('records')
    
    with beam.Pipeline() as pipeline:
        results = (
            pipeline
            | 'Create data' >> beam.Create(data)
            | 'Calculate totals' >> beam.Map(
                lambda x: (x['product_id'], x['quantity'] * x['price']))
            | 'Sum by product' >> beam.CombinePerKey(sum)
            | 'Format for CSV' >> beam.Map(
                lambda x: f"{x[0]},{x[1]:.2f}")
            | 'Write to file' >> beam.io.WriteToText(output_file, shard_name_template='')
        )
    
    with open(output_file, 'r') as f:
        content = f.read()
    with open(output_file, 'w') as f:
        f.write('product_id,total_revenue\n')
        f.write(content)
    
    print(f"CSV processing completed. Results written to {output_file}")

print("Processing CSV files...")
csv_processing_pipeline()

## Exercise 2: JSON File Processing

In [ ]:
import json

def json_processing_pipeline():
    sample_json = '/tmp/sample_data.json'
    sample_data = [
        {"id": 1, "name": "Alice", "score": 85},
        {"id": 2, "name": "Bob", "score": 92},
        {"id": 3, "name": "Charlie", "score": 78}
    ]
    
    with open(sample_json, 'w') as f:
        json.dump(sample_data, f)
    
    with beam.Pipeline() as pipeline:
        (
            pipeline
            | 'Create JSON data' >> beam.Create(sample_data)
            | 'Add grade' >> beam.Map(
                lambda x: {
                    **x,
                    'grade': 'A' if x['score'] >= 90 else 'B' if x['score'] >= 80 else 'C'
                })
            | 'Convert to JSON string' >> beam.Map(json.dumps)
            | 'Write to file' >> beam.io.WriteToText('/tmp/processed_data', shard_name_template='')
        )
    
    with open('/tmp/processed_data', 'r') as f:
        lines = f.readlines()
    
    json_array = '[' + ','.join([line.strip() for line in lines]) + ']'
    with open('/tmp/processed_data.json', 'w') as f:
        f.write(json_array)
    
    print("JSON processing completed. Results written to /tmp/processed_data.json")

print("Processing JSON files...")
json_processing_pipeline()

## Exercise 3: Multiple File Processing

In [ ]:
def multi_file_processing():
    for i in range(3):
        filename = f'/tmp/data_part_{i}.txt'
        with open(filename, 'w') as f:
            for j in range(5):
                f.write(f'file_{i}_line_{j}\n')
    
    with beam.Pipeline() as pipeline:
        (
            pipeline
            | 'Read multiple files' >> beam.io.ReadFromText('/tmp/data_part_*.txt')
            | 'Extract file number' >> beam.Map(
                lambda x: (x.split('_')[1], 1))
            | 'Count per file' >> beam.CombinePerKey(sum)
            | 'Format' >> beam.Map(lambda x: f"File {x[0]}: {x[1]} lines")
            | 'Print' >> beam.Map(print)
        )
    
    print("Multi-file processing completed")

print("Processing multiple files...")
multi_file_processing()

## Summary

This solution demonstrates:
- Text file I/O operations
- CSV file processing and formatting
- JSON data transformation
- Multi-file processing with patterns
- Output formatting and file writing

I/O connectors are essential for integrating Apache Beam pipelines with external data sources and sinks.